# Exploratory Task2 prediction-prior correction (no retraining)

The completed A/B acquisition study improved new-class recall but caused many Benign flows to be predicted as attacks. This notebook evaluates a **hypothesis**: adjusting each fixed checkpoint's logits for the difference between its recorded training-draw class probabilities and original **training-partition** class proportions might reduce Benign→Infiltration false positives. It does not fit a model, change quarantine, or claim probability calibration. Earlier fixed-test results were already inspected, so this is exploratory.

**Locked before scoring:** for each of A and B, use `argmax(logits + strength × log(original_train_prior / sampled_draw_prior))`, with strengths **0, 0.25, 0.5, 0.75, 1.0**. Zero is the uncorrected control. An eligible candidate must meet **Benign FPR ≤ 5%**, **old-class accuracy ≥ 90%**, and **mean recall of Infiltration/Webattack/Portscan ≥ 50%** on full validation. Select the eligible candidate with highest validation macro-F1; exact ties use lower Benign FPR, higher new mean recall, arm name, then strength. If none is eligible, report the tradeoff and **do not open test rows**. Otherwise score only the selected pair on fixed test once. Report all eight classes, not just aggregate accuracy.

Attach (1) the complete prepared-array folder and (2) the complete `task2_clean_acquisition_study` output folder **or** its intact original ZIP. Enable Kaggle Internet and GPU. The original Parquet and Task1 run are not required: prepared arrays and hash-verified A/B checkpoints are sufficient. Keep outputs private, outside Git.


In [ ]:
from pathlib import Path
import subprocess, sys
SOURCE_SHA = '68f815c6291d19b704a412d830949d0d66d279d7'
REPO = Path('/kaggle/working/poison-resilient-continual-nids')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', 'origin', 'feature/unified-novelty'], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO, check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip() == SOURCE_SHA
sys.path.insert(0, str(REPO))
print('Verified source:', SOURCE_SHA)


In [ ]:
import torch
from scripts.run_task2_prior_calibration import ZIP_SHA256, read_verified_acquisition, sha256_file
assert torch.cuda.is_available(), 'Enable a Kaggle GPU and restart'
PREPARED_DIR = Path('/kaggle/input/notebooks/dvdhere/preparation/review2_large_prepare_20261002T223303_224539Z')
ACQUISITION_RESULTS_DIR = None  # Or Path('/kaggle/input/.../task2_clean_acquisition_study')
ACQUISITION_RESULTS_ZIP = None  # Or Path('/kaggle/input/.../results (1).zip')
OUTPUT_DIR = Path('/kaggle/working/task2_prior_calibration_study')
assert PREPARED_DIR.is_dir(), f'Missing complete prepared folder: {PREPARED_DIR}'
if ACQUISITION_RESULTS_DIR is not None and ACQUISITION_RESULTS_ZIP is not None:
    raise ValueError('Set only one explicit acquisition selector')
if ACQUISITION_RESULTS_DIR is not None or ACQUISITION_RESULTS_ZIP is not None:
    source = Path(ACQUISITION_RESULTS_DIR or ACQUISITION_RESULTS_ZIP)
else:
    folders = [p.parent for p in Path('/kaggle/input').rglob('protocol.json') if p.parent.name == 'task2_clean_acquisition_study']
    zips = [p for p in Path('/kaggle/input').rglob('*.zip') if sha256_file(p) == ZIP_SHA256]
    candidates = sorted(set(folders + zips))
    if len(candidates) != 1:
        raise ValueError(f'Expected one completed acquisition folder/ZIP; found {candidates}. Set exactly one explicit selector.')
    source = candidates[0]
read_verified_acquisition(source)  # Full ZIP hash or exact consumed-file hashes; fails closed.
print('GPU:', torch.cuda.get_device_name(0), 'Acquisition evidence:', source)


In [ ]:
# Synthetic correction/selection tests only; no model training or test-data tuning.
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'tests/test_prior_correction.py'], cwd=REPO, check=True)


In [ ]:
command = [sys.executable, 'scripts/run_task2_prior_calibration.py',
           '--prepared-dir', str(PREPARED_DIR),
           '--output-dir', str(OUTPUT_DIR)]
command += (['--acquisition-results-zip', str(source)] if source.is_file()
            else ['--acquisition-results-dir', str(source)])
print('Running inference-only study:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)
print('Review protocol.json, validation_grid_metrics.json, selection_lock.json and findings.json. A test file exists only if a validation-eligible candidate was selected.')
